# Module 00 — Python connection sanity check

Goal: confirm that your `.env`, the Python interpreter, and the running Postgres server are all wired together correctly. No SQL skill required.

Run cells top-to-bottom. If any cell fails, fix it before moving on.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from the repo root
for k in ['PGHOST','PGPORT','PGUSER','PGDATABASE']:
    print(f'{k}={os.environ.get(k)}')

In [ ]:
import psycopg

with psycopg.connect(
    host=os.environ['PGHOST'],
    port=os.environ['PGPORT'],
    user=os.environ['PGUSER'],
    password=os.environ['PGPASSWORD'],
    dbname=os.environ['PGDATABASE'],
) as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT version(), current_database(), current_user, now();')
        print(cur.fetchone())

## Verify the seed loaded

If you skipped the `psql -f seed/schema.sql` step, the next cell will raise. Run the loaders described in `README.md` first.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(os.environ['DATABASE_URL'])
with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT 'users' AS t, count(*) FROM app.users
        UNION ALL SELECT 'posts',       count(*) FROM app.posts
        UNION ALL SELECT 'comments',    count(*) FROM app.comments
        UNION ALL SELECT 'products',    count(*) FROM app.products
        UNION ALL SELECT 'orders',      count(*) FROM app.orders
        UNION ALL SELECT 'order_items', count(*) FROM app.order_items
        UNION ALL SELECT 'inventory',   count(*) FROM app.inventory;
    """), conn)
df

If you got a DataFrame back with the expected counts, you are ready for Module 01.